In [ ]:
from bs4 import BeautifulSoup, Tag
from datetime import datetime
from tqdm import tqdm
import pandas as pd
import requests
from utils.helper_functions import get_path

def parse_industry_html(html: str) -> pd.DataFrame:
    soup = BeautifulSoup(html, "html.parser")
    nav = soup.find("nav", id="subnav") or soup.find("nav", class_="subsec_nav")
    if not nav:
        raise RuntimeError("Could not find nav#subnav in HTML")

    industries = []
    # select anchors under each subLevel block
    for a in nav.select('div.subLevel > a[itemprop="url"], div.subLevel > a'):
        href = a.get("href")
        if not href or href.startswith("javascript"):
            continue
        name = a.get_text(" ", strip=True)
        # normalize relative links
        if href.startswith("/"):
            href = "https://economictimes.indiatimes.com" + href
        industries.append({"industry": name, "link": href})

    return pd.DataFrame(industries)

In [18]:
import re

def parse_industry_html(html: str) -> pd.DataFrame:
    soup = BeautifulSoup(html, "html.parser")
    nav = soup.find("nav", id="subnav") or soup.find("nav", class_=re.compile(r"subsec_nav", re.I))
    if not nav:
        raise RuntimeError("Could not find nav#subnav in HTML")

    industries = []
    seen = set()

    # 1. Grab visible industries (Auto, Banking, Energy, etc.)
    for div in nav.find_all("div", class_=re.compile(r"subLevel", re.I)):
        a = div.find("a")
        if not a:
            continue
        name = a.get_text(" ", strip=True)
        href = a.get("href")
        if not href or href.startswith("javascript") or name == "More":
            continue
        if href.startswith("/"):
            href = "https://economictimes.indiatimes.com" + href
        if href not in seen:
            seen.add(href)
            industries.append({"industry": name, "link": href})

    # 2. Grab dropdown industries from the 'More' menu (Transportation, Tech, Telecom, etc.)
    more_btn = nav.find(string=re.compile(r"^More", re.I))
    if more_btn:
        more_container = more_btn.find_parent("div")
        for a in more_container.find_all("a", class_=re.compile(r"subsec1", re.I)):
            name = a.get_text(" ", strip=True)
            href = a.get("href")
            if not href or href.startswith("javascript"):
                continue
            if href.startswith("/"):
                href = "https://economictimes.indiatimes.com" + href
            if href not in seen:
                seen.add(href)
                industries.append({"industry": name, "link": href})

    return pd.DataFrame(industries)


In [19]:
industry_page_url = "https://m.economictimes.com/industry"
headers = {"User-Agent": "Mozilla/5.0"}
datetime_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

response_industry = requests.get(industry_page_url, headers=headers)

# print(f'Status Code - Industry Page : {response_industry.status_code}')

df_ind = parse_industry_html(response_industry.text)

In [20]:
df_ind.to_csv('test_output.csv')